# Bonus: Implicit Object Representations in `quantEM`

Tutorial by Georgios Varnavides (`G.Varnavides@tudelft.nl`), TU Delft and Stephanie Ribet, LBNL (`sribet@lbl.gov`), adapted from a `quantEM` tutorial by Arthur McCray, Stanforf University.

## What this notebook does

Every iterative reconstruction so far has treated the object as a grid of independent pixels. Each pixel is a free parameter, and nothing in the optimization says that neighbouring pixels ought to look like each other. With a few hundred thousand unknowns and a finite dose, that freedom is expensive: noise in the measurements flows straight into the object.

This notebook replaces that pixel grid with an **implicit representation**. Instead of optimizing pixel values, we optimize the weights of a small convolutional network that *generates* the object. This is the **deep image prior** idea, and the surprise is that it needs no training data at all.

By the end of the notebook you will have:

1. Run a conventional pixel-wise iterative reconstruction as a baseline.
2. Understood what an implicit representation buys you, and what it costs.
3. Initialized a deep image prior model from that baseline and refined it.
4. Compared the two reconstructions side by side.

:::{note}
This is the one notebook of the day that really wants a GPU. In Colab, go to
**Runtime → Change runtime type** and select a **T4 GPU** before running anything.
:::

## 1. Set Up the Environment

In [ ]:
%pip install -q git+https://github.com/electronmicroscopy/quantem.git@dev

In [ ]:
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

import quantem as em
from quantem.core import config
from quantem.core.datastructures import Dataset4dstem
from quantem.diffractive_imaging import PtychographyDatasetRaster, PtychoLite, PtychoLiteDIP

print(f"quantEM version: {em.__version__}")

if torch.cuda.is_available():
    config.set_device(0)
    print(f"Using GPU: {torch.cuda.get_device_name(config.get('device'))}")
elif torch.backends.mps.is_available():
    config.set_device("mps")
    print("No CUDA GPU found, using Apple MPS")
else:
    config.set_device("cpu")
    print("No GPU found, using CPU. This will be slow -- switch to a T4 runtime if you can.")

## 2. Download and Load the Data

A simulated 4D-STEM scan of a cartoon duck embedded in a graphene substrate, at 80 keV with a 20 mrad convergence semi-angle, 500 Å of defocus, a 4 Å scan step, and a finite dose of 5 × 10$^4$ e$^-$/Å$^2$ already applied.

The defocused probe overlaps heavily between neighbouring scan positions, which is exactly the redundancy iterative ptychography feeds on. The object is a cartoon rather than a crystal on purpose: it has sharp edges, flat regions, and fine detail all at once, so it is easy to see which of those a given reconstruction preserves.

In [ ]:
import os
import gdown

dirpath = "/content/"
filepath_data = dirpath + "ducky_251105_20mrad_500A-df_4A-step_5e+04-dose_clean.zip"

if not os.path.exists(filepath_data):
    gdown.download(
        id="1vA0DXoI9qPkdY_ImfBhPLW6zBVZgeLoY",
        output=filepath_data,
        quiet=False,
    )

In [ ]:
dataset: Dataset4dstem = em.io.load(filepath_data)
dataset

In [ ]:
PROBE_ENERGY = 80e3      # eV
PROBE_SEMIANGLE = 20     # mrad
PROBE_DEFOCUS = 500      # Angstrom

## 3. Preprocess the Scan Geometry

Iterative ptychography needs the scan positions in the same coordinate frame as the diffraction patterns, so the first step estimates the scan-to-detector rotation from the center-of-mass signal, exactly the calibration we met earlier in the day. The plots below are the diagnostic: the correct rotation is the one that makes the center-of-mass field curl-free.

In [ ]:
pdset = PtychographyDatasetRaster.from_dataset4dstem(dataset)

pdset.preprocess(
    com_fit_function="constant",
    plot_rotation=True,
    plot_com=True,
)

## 4. The Baseline: A Pixel-Wise Reconstruction

Build a standard `PtychoLite` reconstruction. `obj_type="pure_phase"` says the sample is a weak phase object that absorbs nothing, which is a reasonable model here and removes half the unknowns.

Note what the object actually *is* at this point: a dense array of free parameters, one per pixel, optimized by gradient descent against the measured patterns.

In [ ]:
ptycho_pix = PtychoLite.from_dataset(
    dset=pdset,
    num_slices=1,
    num_probes=1,
    obj_type="pure_phase",
    energy=PROBE_ENERGY,
    defocus=PROBE_DEFOCUS,
    semiangle_cutoff=PROBE_SEMIANGLE,
    obj_padding_px=(32, 32),
    device=config.get_device()
)

print("object array shape:", tuple(ptycho_pix.obj.shape))
print("object sampling [A]:", np.round(np.asarray(ptycho_pix.sampling), 3))

In [ ]:
ptycho_pix.reconstruct(
    num_iters=50,
    reset=True,
    lr_obj=5e-2,
    lr_probe=5e-2,
    batch_size=125,
    scheduler_type="plateau",
).visualize()

Continue from where we left off. Passing `reset=False`, the default, resumes rather than restarting.

In [ ]:
ptycho_pix.reconstruct(num_iters=150).visualize()

In [ ]:
ptycho_pix.show_obj(
    axsize=(8, 8),
    cmap="turbo",
    title=f"pixel-wise, {ptycho_pix.num_iters} iterations"
)

The duck is clearly there. Look closely at the flat regions, though: they are grainy, and the graininess is not sample structure, it is measurement noise being faithfully reproduced by an object model with enough freedom to fit it.

This is the fundamental tension in a pixel-wise parameterization. More iterations reduce the data misfit, but past a point they are fitting noise rather than sample. The usual remedy is explicit regularization, a smoothness penalty or a total-variation term, which means choosing a functional form for "what objects look like" and a weight to trade it against the data.

## 5. Why an Implicit Representation?

A deep image prior takes a different route. Rather than optimizing the object pixels $O$ directly, we write

$$ O = f_\theta(z) $$

where $z$ is a *fixed* random input and $f_\theta$ is a small convolutional network. The optimization now runs over the network weights $\theta$, not over pixels.

The striking result, from Ulyanov and co-workers, is that this regularizes even though the network is never trained on anything. The prior is **structural**: a convolutional generator, by its architecture, reaches smooth and self-similar images quickly and noisy ones only slowly. Stop the optimization at a sensible point and you get the signal without the noise. Nothing was learned from a dataset of other samples, which matters a great deal in electron microscopy, where such a dataset usually does not exist and would bias the result if it did.

Three practical consequences worth holding onto:

- **No training data, no pretrained weights.** The network is fit to this one measurement, from scratch.
- **Regularization without choosing a penalty.** The architecture is the prior. You are still making a choice, but it is a choice about representation rather than about a hand-tuned weight.
- **It is not free.** Each iteration now involves a forward and backward pass through a network, so iterations are several times more expensive, and there are new hyperparameters (network depth, learning rates) to get wrong.

A good workflow, and the one below, is to start the network from a partially converged pixel-wise solution rather than from noise.

## 6. Initialize the Deep Image Prior

First make a copy of the reconstruction and run only a handful of pixel-wise iterations, so the comparison at the end is instructive rather than a foregone conclusion.

In [ ]:
ptycho_pix_short = PtychoLite.from_ptychography(ptycho=ptycho_pix)
ptycho_pix_short.device=config.get_device()

ptycho_pix_short.reconstruct(
    num_iters=5,
    reset=True,
    lr_obj=5e-2,
    lr_probe=5e-2,
).visualize()

Now build the DIP model from it. `pretrain_iters` controls a short warm-up phase in which the network learns to reproduce the current object and probe, so that the reconstruction starts from a sensible place in weight space rather than from a random image.

`cnn_num_layers` sets the depth of the generator, and is the main knob on how strong the structural prior is.

In [ ]:
ptycho_dip = PtychoLiteDIP.from_ptycholite(
    ptycholite=ptycho_pix_short,
    pretrain_iters=50,
    cnn_num_layers=3,
    device=config.get_device()
)

## 7. Refine Through the Network

From here the loop looks identical to the pixel-wise case: forward-model the diffraction patterns, compare against the measurement, backpropagate. The only difference is that the gradients now flow through the generator into its weights.

Note the much smaller learning rates. We are stepping in weight space, where a small change can move every object pixel at once.

In [ ]:
ptycho_dip.reconstruct(
    num_iters=15,
    reset=True,
    lr_obj=5e-4,
    lr_probe=5e-4,
    batch_size=125,
    scheduler_type="plateau",
).visualize()

In [ ]:
ptycho_dip.reconstruct(num_iters=15).visualize()

## 8. Compare

Put the two reconstructions side by side. The pixel-wise result has had an order of magnitude more iterations, so this is not a fair fight on compute, which makes the comparison more interesting rather than less.

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(16, 8))

ptycho_pix.show_obj(
    figax=(fig, axs[0]), cmap="turbo",
    title=f"pixel-wise, {ptycho_pix.num_iters} iterations",
)
ptycho_dip.show_obj(
    figax=(fig, axs[1]), cmap="turbo",
    title=f"deep image prior, {ptycho_dip.num_iters} iterations",
)
plt.show()

What to look for, in roughly this order:

1. **Flat regions.** The DIP result should be visibly cleaner where the sample is featureless, without a smoothness penalty ever having been specified.
2. **Edges.** Check that the duck's outline is still sharp. If the prior were simply blurring the object, edges would soften along with the noise. They should not.
3. **Fine detail.** This is where to be sceptical. A structural prior that suppresses noise can also suppress genuine low-contrast detail, and there is no line in the output telling you which happened. Compare against the pixel-wise result before believing any small feature.

## 9. Saving and Resuming

Long reconstructions are worth checkpointing, and DIP runs are longer than most.

In [ ]:
savedir = Path("/content/outputs")
savedir.mkdir(parents=True, exist_ok=True)

ptycho_dip.save(savedir / "ptycholite_dip.zip", mode="o")

In [ ]:
ptycho_dip_reloaded = PtychoLite.from_file(savedir / "ptycholite_dip.zip", dset=pdset)
ptycho_dip_reloaded.device = config.get_device()
ptycho_dip_reloaded.reconstruct(num_iters=10).visualize()

## 10. What to Notice and What to Try Next

The summary:

- An implicit representation regularizes through the *structure* of the generator, not through a penalty term you have to weight.
- It needs no training data, which is what makes it usable on a sample nobody has imaged before.
- Iterations cost more, so it earns its keep at low dose, where the pixel-wise reconstruction is noise-limited rather than iteration-limited.
- It is a prior, and priors can be wrong. Treat faint features as unconfirmed until a pixel-wise reconstruction agrees.

Follow-up exercises:

1. Vary `cnn_num_layers` between 2 and 5 and watch how the strength of the prior changes.
2. Skip the pretraining step, with `pretrain_iters=0`, and see how much harder the optimization becomes.
3. Run both reconstructions much longer and look for the point where the pixel-wise result starts visibly overfitting the noise while the DIP result does not.
4. Reduce the dose by resampling the dataset with `np.random.poisson`, as in the earlier notebooks, and repeat the comparison. This is the regime the method is really for.
5. Switch `obj_type` from `"pure_phase"` to `"complex"` and see what the extra freedom costs you.

## References

[1] D. Ulyanov, A. Vedaldi, and V. Lempitsky, "Deep Image Prior," *CVPR* (2018). DOI: <https://doi.org/10.1109/CVPR.2018.00984>.

[2] A. R. C. McCray, S. M. Ribet, G. Varnavides, and C. Ophus, "Deep generative priors for robust and efficient electron ptychography," *arXiv* (2025). DOI: <https://doi.org/10.48550/arXiv.2511.07795>.

[3] `quantEM` documentation: <https://electronmicroscopy.github.io/quantem-docs/>